# Build A Dataset In Colab

This notebook is the Colab path for `faster-whisper` dataset generation.

Workflow:

1. mount Google Drive
2. point the notebook at a folder with source audio
3. optionally normalize audio with `prepare`
4. build the dataset with `sherpa-tts dataset`
5. review `sources.csv`, `rejected.csv`, and `metadata.csv`

## 0. Select A GPU Runtime

In Colab: `Runtime -> Change runtime type -> T4 GPU` or any available GPU.

The notebook can run on CPU too, but dataset generation will usually be slower.

In [ ]:
import subprocess
import sys

print("Python:", sys.version.split()[0])
subprocess.run(["nvidia-smi"], check=False)


## 1. Edit This Cell Only

Set your Drive paths and choose where `dataset` should read audio from:

- `raw`: use `RAW_AUDIO_DIR` directly
- `prepared`: use an already prepared folder
- `prepare-first`: run `prepare` and then build the dataset from the prepared output

In [ ]:
REPO_URL = "https://github.com/kymuco/sherpa-tts-pipeline.git"
REPO_BRANCH = "main"

RAW_AUDIO_DIR = "/content/drive/MyDrive/sherpa_tts/raw_audio/my_voice"
DATASET_SOURCE_MODE = "raw"  # raw | prepared | prepare-first
PREPARED_AUDIO_DIR = "/content/drive/MyDrive/sherpa_tts/prepared_audio/my_voice"  # used only for prepared or prepare-first
OUTPUT_DATASET_DIR = "/content/drive/MyDrive/sherpa_tts/data/my_voice"

CONFIG_MODE = "default"  # default | rescue | custom | none
CUSTOM_CONFIG_PATH = ""
WHISPER_MODEL_OVERRIDE = ""

OVERWRITE = False
APPEND = False
VERBOSE = True


## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. Clone The Repo And Install Dataset Dependencies

In [ ]:
from pathlib import Path
import shlex
import shutil
import subprocess
import sys


def run(cmd, cwd=None):
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print("$", printable)
    subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, check=True)


WORKDIR = Path("/content")
REPO_DIR = WORKDIR / "sherpa-tts-pipeline"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
run(["apt-get", "update", "-y"])
run(["apt-get", "install", "-y", "ffmpeg"])
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "PyYAML>=6.0",
    "ctranslate2>=4.0.0",
    "faster-whisper>=1.1.0",
    "soundfile>=0.12.1",
])
run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], cwd=REPO_DIR)


def sherpa_cmd(subcommand, *extra_args):
    cmd = [sys.executable, "-m", "sherpa_tts_pipeline.cli"]
    if VERBOSE:
        cmd.append("--verbose")
    cmd.append(subcommand)
    cmd.extend(str(arg) for arg in extra_args)
    return cmd


print("Repo:", REPO_DIR)


## 4. Resolve Paths And Config

In [ ]:
from pathlib import Path

raw_audio_dir = Path(RAW_AUDIO_DIR)
source_mode = DATASET_SOURCE_MODE.strip().lower()
prepared_audio_dir = Path(PREPARED_AUDIO_DIR) if PREPARED_AUDIO_DIR.strip() else None
output_dataset_dir = Path(OUTPUT_DATASET_DIR)
output_dataset_dir.parent.mkdir(parents=True, exist_ok=True)

config_map = {
    "default": REPO_DIR / "examples" / "voice.yaml",
    "rescue": REPO_DIR / "examples" / "voice_rescue.yaml",
}

if CONFIG_MODE == "custom":
    if not CUSTOM_CONFIG_PATH.strip():
        raise ValueError("CUSTOM_CONFIG_PATH must be set when CONFIG_MODE='custom'.")
    config_path = Path(CUSTOM_CONFIG_PATH)
elif CONFIG_MODE == "none":
    config_path = None
elif CONFIG_MODE in config_map:
    config_path = config_map[CONFIG_MODE]
else:
    raise ValueError(f"Unsupported CONFIG_MODE: {CONFIG_MODE}")

if not raw_audio_dir.exists():
    raise FileNotFoundError(f"RAW_AUDIO_DIR not found: {raw_audio_dir}")

if source_mode not in {"raw", "prepared", "prepare-first"}:
    raise ValueError(
        "DATASET_SOURCE_MODE must be one of: raw, prepared, prepare-first"
    )

if source_mode in {"prepared", "prepare-first"} and prepared_audio_dir is None:
    raise ValueError(
        "PREPARED_AUDIO_DIR must be set when DATASET_SOURCE_MODE is 'prepared' or 'prepare-first'."
    )

if source_mode == "prepared" and not prepared_audio_dir.exists():
    raise FileNotFoundError(f"PREPARED_AUDIO_DIR not found: {prepared_audio_dir}")

if config_path is not None and not config_path.exists():
    raise FileNotFoundError(f"Config not found: {config_path}")

print("Source mode:", source_mode)
print("Raw audio:", raw_audio_dir)
if prepared_audio_dir is not None:
    print("Prepared audio:", prepared_audio_dir)
print("Output dataset:", output_dataset_dir)
print("Config:", config_path or "<none>")


## 5. Resolve Dataset Source

In [ ]:
if source_mode == "prepare-first":
    prepared_audio_dir.parent.mkdir(parents=True, exist_ok=True)
    cmd = sherpa_cmd("prepare", raw_audio_dir, "--out", prepared_audio_dir)
    if config_path is not None:
        cmd.extend(["--config", config_path])
    run(cmd)
    dataset_source_dir = prepared_audio_dir
elif source_mode == "prepared":
    print("Using PREPARED_AUDIO_DIR as the dataset source.")
    dataset_source_dir = prepared_audio_dir
else:
    print("Using RAW_AUDIO_DIR directly. No prepare stage will run.")
    dataset_source_dir = raw_audio_dir

print("Dataset source:", dataset_source_dir)


## 6. Build The Dataset

In [ ]:
cmd = sherpa_cmd("dataset", dataset_source_dir, "--out", output_dataset_dir)

if config_path is not None:
    cmd.extend(["--config", config_path])

if WHISPER_MODEL_OVERRIDE.strip():
    cmd.extend(["--whisper-model", WHISPER_MODEL_OVERRIDE.strip()])

if OVERWRITE:
    cmd.append("--overwrite")

if APPEND:
    cmd.append("--append")

run(cmd)


## 7. Inspect The Result

In [ ]:
from pathlib import Path
import pandas as pd

dataset_dir = Path(OUTPUT_DATASET_DIR)
sources_path = dataset_dir / "sources.csv"
clips_path = dataset_dir / "clips.csv"
rejected_path = dataset_dir / "rejected.csv"
metadata_path = dataset_dir / "metadata.csv"

for path in [sources_path, clips_path, rejected_path]:
    if path.exists():
        df = pd.read_csv(path)
        print(f"\n{path.name}: {len(df)} rows")
        display(df.head(10))
    else:
        print(f"\nMissing: {path}")

if metadata_path.exists():
    metadata_df = pd.read_csv(metadata_path, sep="|", names=["clip_id", "text"])
    print(f"\nmetadata.csv: {len(metadata_df)} rows")
    display(metadata_df.head(10))
else:
    print(f"\nMissing: {metadata_path}")


## Next Step

Once the dataset looks good, open `notebooks/train_piper_colab.ipynb` and train from the generated `metadata.csv` and `wavs/` directory.